In [5]:
import os
from dotenv import load_dotenv
from langfuse import get_client
from openinference.instrumentation.smolagents import SmolagentsInstrumentor
from smolagents import CodeAgent, LiteLLMModel, GoogleSearchTool, VisitWebpageTool

load_dotenv(override=True)

# Tracing
langfuse = get_client()
print("Langfuse ready:", langfuse.auth_check())
SmolagentsInstrumentor().instrument()

# Shared model for both agents
model = LiteLLMModel(
    model_id="gemini/gemini-3.5-flash-lite",
    api_key=os.environ["GEMINI_API_KEY"],
    timeout=60,
    num_retries=0,
)

model2 = LiteLLMModel(
    model_id="ollama_chat/qwen2.5-coder:14b",
    api_base="http://localhost:11434",
    num_ctx=16384,
)

Attempting to instrument while already instrumented


Langfuse ready: True


Initialize the sub agent who browse and retrieve information from internet websites

In [6]:
# Sub-agent: only browses the web, no extra imports
web_agent = CodeAgent(
    model=model,
    tools=[
        GoogleSearchTool(provider="serper"),
        VisitWebpageTool(max_output_length=5000),
        ],
    name="web_agent",
    description=(
        "Searches the web and reads webpages."
        "Use it for any information that must be found online."
        "Give it a precise, self-contained question: it has no access to the manager's context."
    ),
    max_steps=4,
)

Initialize the manager of the sub-agent

In [7]:
# Manager: delegates web lookups, never browses itself
manager_agent = CodeAgent(
    model=model,
    tools=[],
    managed_agents=[web_agent],
    max_steps=3,
    # Sub-agent calls run inside the manager's code step: allow time for a full web_agent run
    executor_kwargs={"timeout_seconds": 600},
)

# Check the multi-agent setup
manager_agent.visualize()

CodeAgent | gemini/gemini-3.5-flash-lite
├── ✅ Authorized imports: []
├── 🛠️ Tools:
│   ┏━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
│   ┃ Name         ┃ Description                                   ┃ Arguments                                    ┃
│   ┡━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│   │ final_answer │ Provides a final answer to the given problem. │ answer (`any`): The final answer to the      │
│   │              │                                               │ problem                                      │
│   └──────────────┴───────────────────────────────────────────────┴──────────────────────────────────────────────┘
└── 🤖 Managed agents:
    └── web_agent | CodeAgent | gemini/gemini-3.5-flash-lite
        ├── ✅ Authorized imports: []
        ├── 📝 Description: Searches the web and reads webpages.Use it for any information that must be found 
        │   online.Give it a precise, self-contained question: it has no access to the manager's context.
        └── 🛠️ Tools:
            ┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
            ┃ Name          ┃ Description                              ┃ Arguments                                ┃
            ┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
            │ web_search    │ Performs a google web search for your    │ query (`string`): The search query to    │
            │               │ query then returns a string of the top   │ perform.                                 │
            │               │ search results.                          │ filter_year (`integer`): Optionally      │
            │               │                                          │ restrict results to a certain year       │
            │ visit_webpage │ Visits a webpage at the given url and    │ url (`string`): The url of the webpage   │
            │               │ reads its content as a markdown string.  │ to visit.                                │
            │               │ Use this to browse webpages.             │                                          │
            │ final_answer  │ Provides a final answer to the given     │ answer (`any`): The final answer to the  │
            │               │ problem.                                 │ problem                                  │
            └───────────────┴──────────────────────────────────────────┴──────────────────────────────────────────┘

In [8]:
result = manager_agent.run(
    "Find 3 private banks headquartered in Geneva and give the official website of each."
)
print(result)

langfuse.flush()

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Find 3 private banks headquartered in Geneva and give the official website of each.                             │
│                                                                                                                 │
╰─ LiteLLMModel - gemini/gemini-3.5-flash-lite ───────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  task_desc = "Find 3 private banks headquartered in Geneva, Switzerland, and provide the official website of      
  each. Return the bank names and their official website URLs."                                                    
  result = web_agent(task=task_desc, additional_args={})                                                           
  print(result)                                                                                                    
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

╭────────────────────────────────────────────── New run - web_agent ──────────────────────────────────────────────╮
│                                                                                                                 │
│ You're a helpful agent named 'web_agent'.                                                                       │
│ You have been submitted this task by your manager.                                                              │
│ ---                                                                                                             │
│ Task:                                                                                                           │
│ Find 3 private banks headquartered in Geneva, Switzerland, and provide the official website of each. Return the │
│ bank names and their official website URLs.                                                                     │
│ ---                                                                                                             │
│ You're helping your manager solve a wider task: so make sure to not provide a one-line answer, but give as much │
│ information as possible to give them a clear understanding of the answer.                                       │
│                                                                                                                 │
│ Your final_answer WILL HAVE to contain these parts:                                                             │
│ ### 1. Task outcome (short version):                                                                            │
│ ### 2. Task outcome (extremely detailed version):                                                               │
│ ### 3. Additional context (if relevant):                                                                        │
│                                                                                                                 │
│ Put all these in your final_answer tool, everything that you do not pass as an argument to final_answer will be │
│ lost.                                                                                                           │
│ And even if your task resolution is not successful, please return as much context as possible, so that your     │
│ manager can act upon this feedback.                                                                             │
│                                                                                                                 │
╰─ LiteLLMModel - gemini/gemini-3.5-flash-lite ───────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  search_results = web_search(query="private banks headquartered in Geneva Switzerland")                           
  print(search_results)                                                                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

... (17 outputs left out)

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("""1. Lombard Odier - https://www.lombardodier.com/                                                 
  2. Pictet Group - https://www.pictet.com/                                                                        
  3. Mirabaud Group - https://www.mirabaud.com/""")                                                                
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 1. Lombard Odier - https://www.lombardodier.com/
2. Pictet Group - https://www.pictet.com/
3. Mirabaud Group - https://www.mirabaud.com/

[Step 2: Duration 8.55 seconds| Input tokens: 5,551 | Output tokens: 122]

1. Lombard Odier - https://www.lombardodier.com/
2. Pictet Group - https://www.pictet.com/
3. Mirabaud Group - https://www.mirabaud.com/


## Notes

- 14b model: web_agent skipped the search and invented an answer (Credit Suisse, UBS...), while saying it had searched. Only the Langfuse trace and Serper logs show what really ran.
- Manager has a 30s limit per code step, and the web_agent call runs inside it. Answers got dropped and it looped. Fixed with `executor_kwargs={"timeout_seconds": 600}` on the manager.
- gemini-3.5-flash: 503 / timeouts. flash-lite works.